# 04 - BM25 + Query Expansion

Notebook này xây dựng phương pháp truy xuất **BM25 + Query Expansion**.

Luồng chính:

```text
news_processed.pkl
→ tokens
→ BM25 Index thủ công
→ Search lần 1
→ Query Expansion từ top documents
→ Search lần 2
→ Top-k kết quả
```

## Cell 1 - Import thư viện

Cell này import các thư viện cần thiết để:
- Đọc dữ liệu đã xử lý.
- Tự xây BM25 index.
- Lưu index/model.

In [ ]:
import os
import math
import pickle
import pandas as pd
import numpy as np

from collections import Counter, defaultdict

## Cell 2 - Import hàm xử lý query dùng chung

Cả 3 phương pháp dùng chung `preprocess_query()` trong `src/preprocess.py`.

In [ ]:
import sys

CURRENT_DIR = os.getcwd()

if os.path.basename(CURRENT_DIR) == "notebooks":
    PROJECT_ROOT = os.path.abspath("..")
else:
    PROJECT_ROOT = CURRENT_DIR

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from src.preprocess import preprocess_query

## Cell 3 - Khai báo đường dẫn

Cell này khai báo:
- File dữ liệu đã xử lý.
- Folder lưu index/model của phương pháp BM25 + Query Expansion.

In [ ]:
DATA_PATH = "../data/processed/news_processed.pkl"

MODEL_DIR = "../models/bm25_query_expansion"
os.makedirs(MODEL_DIR, exist_ok=True)

## Cell 4 - Load dữ liệu đã xử lý

Dữ liệu đầu vào là `news_processed.pkl`, dùng chung với các phương pháp khác.

In [ ]:
df = pd.read_pickle(DATA_PATH)

print("Shape:", df.shape)
print(df.columns.tolist())

df[["doc_id", "title", "topic", "source", "processed_text"]].head()

## Cell 5 - Lấy tokenized corpus

BM25 làm việc trực tiếp với danh sách token. Nếu cột `tokens` có sẵn thì dùng `tokens`, nếu không thì tách từ `processed_text`.

In [ ]:
if "tokens" in df.columns:
    tokenized_corpus = df["tokens"].tolist()
else:
    tokenized_corpus = df["processed_text"].fillna("").astype(str).apply(lambda x: x.split()).tolist()

N = len(tokenized_corpus)

print("Số documents:", N)
print(tokenized_corpus[0][:50])

## Cell 6 - Tính độ dài document

BM25 cần:
- Độ dài từng document.
- Độ dài trung bình của toàn bộ corpus.

In [ ]:
doc_lengths = np.array([len(tokens) for tokens in tokenized_corpus], dtype=np.float32)
avg_doc_length = doc_lengths.mean()

print("Average document length:", avg_doc_length)

## Cell 7 - Xây inverted index cho BM25

Inverted index có dạng:

```text
token → {doc_id: term_frequency}
```

Cấu trúc này giúp BM25 chỉ tính điểm trên các document chứa token trong query.

In [ ]:
inverted_index = defaultdict(dict)
df_counter = Counter()

for doc_idx, tokens in enumerate(tokenized_corpus):
    token_counts = Counter(tokens)

    for token, tf in token_counts.items():
        inverted_index[token][doc_idx] = tf

    df_counter.update(token_counts.keys())

print("Số token trong inverted index:", len(inverted_index))

## Cell 8 - Tính IDF cho BM25

Công thức IDF của BM25:

```text
IDF(t) = log(1 + (N - DF(t) + 0.5) / (DF(t) + 0.5))
```

In [ ]:
bm25_idf = {}

for token, df_t in df_counter.items():
    bm25_idf[token] = math.log(1 + (N - df_t + 0.5) / (df_t + 0.5))

print("Số token có IDF:", len(bm25_idf))

## Cell 9 - Tạo metadata

Metadata dùng để hiển thị kết quả truy xuất.

In [ ]:
metadata = df[[
    "doc_id",
    "id",
    "title",
    "content",
    "topic",
    "source",
    "url"
]].copy()

metadata.head()

## Cell 10 - Lưu BM25 index

Cell này lưu các thành phần cần thiết để search lại:
- `inverted_index.pkl`
- `bm25_idf.pkl`
- `doc_lengths.pkl`
- `metadata.pkl`

In [ ]:
with open(f"{MODEL_DIR}/inverted_index.pkl", "wb") as f:
    pickle.dump(dict(inverted_index), f)

with open(f"{MODEL_DIR}/bm25_idf.pkl", "wb") as f:
    pickle.dump(bm25_idf, f)

with open(f"{MODEL_DIR}/doc_lengths.pkl", "wb") as f:
    pickle.dump(doc_lengths, f)

with open(f"{MODEL_DIR}/metadata.pkl", "wb") as f:
    pickle.dump(metadata, f)

with open(f"{MODEL_DIR}/avg_doc_length.pkl", "wb") as f:
    pickle.dump(avg_doc_length, f)

print("Đã lưu BM25 index vào:", MODEL_DIR)

## Cell 11 - Hàm tính điểm BM25

BM25 tính điểm dựa trên:
- TF của token trong document.
- IDF của token.
- Độ dài document.
- Độ dài trung bình của corpus.

In [ ]:
def bm25_score(query_tokens, k1=1.5, b=0.75):
    scores = defaultdict(float)

    for token in query_tokens:
        if token not in inverted_index:
            continue

        idf = bm25_idf.get(token, 0.0)
        posting = inverted_index[token]

        for doc_idx, tf in posting.items():
            dl = doc_lengths[doc_idx]

            numerator = tf * (k1 + 1)
            denominator = tf + k1 * (1 - b + b * dl / avg_doc_length)

            scores[doc_idx] += idf * numerator / denominator

    return scores

## Cell 12 - Hàm search BM25 cơ bản

Hàm này:
1. Xử lý query.
2. Tính BM25 score.
3. Trả về top-k document có điểm cao nhất.

In [ ]:
def search_bm25(query, top_k=10):
    query_processed, query_tokens = preprocess_query(query)

    scores_dict = bm25_score(query_tokens)

    if len(scores_dict) == 0:
        return pd.DataFrame(columns=list(metadata.columns) + ["score", "query_processed"])

    ranked = sorted(scores_dict.items(), key=lambda x: x[1], reverse=True)[:top_k]

    top_indices = [doc_idx for doc_idx, score in ranked]
    top_scores = [score for doc_idx, score in ranked]

    results = metadata.iloc[top_indices].copy()
    results["score"] = top_scores
    results["query_processed"] = query_processed

    return results

## Cell 13 - Hàm lấy từ mở rộng query

Query Expansion lấy thêm các token nổi bật từ top document của lần search đầu tiên.

Ở đây dùng cách đơn giản:
- Lấy top-k document đầu.
- Đếm tần suất token trong các document đó.
- Loại token đã có trong query.
- Lấy `expand_n` token phổ biến nhất để mở rộng query.

In [ ]:
def get_expansion_terms(query_tokens, top_doc_indices, expand_n=5):
    query_token_set = set(query_tokens)

    token_counter = Counter()

    for doc_idx in top_doc_indices:
        token_counter.update(tokenized_corpus[doc_idx])

    expansion_terms = []

    for token, count in token_counter.most_common():
        if token not in query_token_set and token in bm25_idf:
            expansion_terms.append(token)

        if len(expansion_terms) >= expand_n:
            break

    return expansion_terms

## Cell 14 - Hàm search BM25 + Query Expansion

Quy trình:
1. Search BM25 lần 1.
2. Lấy top document làm nguồn mở rộng query.
3. Thêm từ mở rộng vào query.
4. Search BM25 lần 2.

In [ ]:
def search_bm25_query_expansion(
    query,
    top_k=10,
    feedback_pool=10,
    expand_n=5
):
    query_processed, query_tokens = preprocess_query(query)

    first_scores = bm25_score(query_tokens)

    if len(first_scores) == 0:
        return pd.DataFrame(columns=list(metadata.columns) + ["score", "query_processed", "expanded_query"])

    first_ranked = sorted(first_scores.items(), key=lambda x: x[1], reverse=True)[:feedback_pool]
    top_doc_indices = [doc_idx for doc_idx, score in first_ranked]

    expansion_terms = get_expansion_terms(
        query_tokens=query_tokens,
        top_doc_indices=top_doc_indices,
        expand_n=expand_n
    )

    expanded_query_tokens = query_tokens + expansion_terms
    expanded_query = " ".join(expanded_query_tokens)

    second_scores = bm25_score(expanded_query_tokens)
    second_ranked = sorted(second_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

    top_indices = [doc_idx for doc_idx, score in second_ranked]
    top_scores = [score for doc_idx, score in second_ranked]

    results = metadata.iloc[top_indices].copy()
    results["score"] = top_scores
    results["query_processed"] = query_processed
    results["expanded_query"] = expanded_query

    return results

## Cell 15 - Test BM25 cơ bản

Cell này chạy thử BM25 chưa có Query Expansion.

In [ ]:
query = "cầu thủ Quang Hải"

results = search_bm25(query, top_k=10)

print("Query gốc:", query)
print("Query sau xử lý:", results["query_processed"].iloc[0] if len(results) > 0 else "")

results[[
    "doc_id",
    "title",
    "topic",
    "source",
    "url",
    "score"
]]

## Cell 16 - Test BM25 + Query Expansion

Cell này chạy thử BM25 sau khi mở rộng query.

In [ ]:
query = "cầu thủ Quang Hải"

results = search_bm25_query_expansion(query, top_k=10, feedback_pool=10, expand_n=5)

print("Query gốc:", query)

if len(results) > 0:
    print("Query sau xử lý:", results["query_processed"].iloc[0])
    print("Query mở rộng:", results["expanded_query"].iloc[0])

results[[
    "doc_id",
    "title",
    "topic",
    "source",
    "url",
    "score"
]]

## Cell 17 - Hiển thị kết quả dạng dễ đọc

Cell này in kết quả theo từng rank để dễ kiểm tra nội dung bài báo.

In [ ]:
def show_results(results):
    for rank, (_, row) in enumerate(results.iterrows(), start=1):
        print("=" * 100)
        print(f"Rank {rank}")
        print("Score :", round(row["score"], 4))
        print("Title :", row["title"])
        print("Topic :", row["topic"])
        print("Source:", row["source"])
        print("URL   :", row["url"])
        if "expanded_query" in results.columns:
            print("Expanded query:", row.get("expanded_query", ""))
        print("Content preview:")
        print(str(row["content"])[:500])

In [ ]:
show_results(results)